# Chapter 6 — Specialized Agentic EDA for Analog Design

**Multi-Agent Analog EDA — PhD-Level Notebook**

---

This chapter surveys **state-of-the-art (SOTA) agentic frameworks** targeting *analog* and *mixed-signal* flows: **code-generating simulators**, **self-evolving stratified memory**, **training-free structural inference** from SPICE netlists, **schematic bridging** for human–machine communication, and **collaborative layout** agents. Where proprietary weights or cloud APIs are unavailable, we provide **self-contained implementations** that preserve the *control topology* and *verification interfaces* emphasized in recent papers.

### Learning objectives

1. **Formalize** analog design as **conditional program synthesis** $\pi_\theta(c \mid s)$ over simulator APIs (AnalogCoder) and connect **multimodal waveform critique** (AnalogCoder-Pro).
2. **Implement** a **stratified memory hierarchy** (Evolution / Introspective / Fusion) and study its growth across iterations (AnalogSAGE).
3. **Identify** functional subcircuits in SPICE via **graph-pattern detectors** in the spirit of GENIE-ASI (training-free structural priors + few-shot code).
4. **Sketch** the **netlist $\rightarrow$ LTspice `.asc`** pipeline (Schemato) as learned layout over symbolic schematic space.
5. **Simulate** **multi-agent layout collaboration** with shared violation signals (LayoutCopilot).

### Notation

- Structured specification $s \in \mathcal{S}$ (targets, corners, PDK hooks).
- Executable simulator script $c \in \mathcal{C}$ (PySpice / Ngspice / Spectre-in-Python wrappers).
- Oracle $\mathcal{O}(c) \mapsto (y, \phi)$: time-domain / frequency-domain waveforms $y$ and figures-of-merit $\phi$.
- Stratified memories $\mathcal{M}_E, \mathcal{M}_I, \mathcal{M}_F$.

---


In [ ]:
# Global imports & dark theme (self-contained; no API keys)
from __future__ import annotations

import hashlib
import json
import re
import textwrap
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

RNG = np.random.default_rng(2026)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
AMBER = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"

MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.alpha": 0.65,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)
pio.templates.default = "plotly_dark"

print("Ready: numpy, matplotlib (#0d1117), plotly (plotly_dark).")


## 6.1 AnalogCoder & AnalogCoder-Pro — Programs as Analog Artifacts

### 6.1.1 Code generation as a design policy

**AnalogCoder** treats analog sizing + verification as **search in a program space** $\mathcal{C}$ rather than only in a continuous parameter vector. Given structured specs $s$, a policy (LLM or hybrid) outputs a script $c$ that constructs netlists, sources, and analyses:

$$
c \sim \pi_\theta(c \mid s), \qquad \hat{y} = \mathcal{S}(c).
$$

A **spec-matching objective** can be written as

$$
\mathcal{L}(c; s) = \sum_k w_k \left\lvert \phi_k(\hat{y}) - \phi_k^\star(s) \right\rvert^p + \lambda \, \rho(c),
$$

where $\phi_k$ are simulator-derived metrics (gain, PM, noise, PSRR), $\phi_k^\star$ targets, and $\rho$ penalizes unsafe or non-idiomatic code (undeclared nodes, missing `.ends`, disallowed imports).

### 6.1.2 AnalogCoder-Pro: multimodal grounding on $y(t)$

**AnalogCoder-Pro** augments text generation with **waveform understanding**: an encoder $e_\psi$ maps traces to latents concatenated with textual $s$, enabling **visual** diagnosis (harmonics, clipping, ringing). A generic form:

$$
z = g_\psi\bigl( \mathrm{Tok}(s) \parallel e_\psi(y) \bigr), \qquad \pi_\theta(c \mid s, z).
$$

In practice $e_\psi$ may operate on **spectrograms**, **wavelet coefficients**, or **hand-crafted physics features** (THD, overshoot, encroachment on supply rails)—we demonstrate the latter *without* any external API.

---


In [ ]:
# --- Simplified AnalogCoder: spec -> PySpice-style Python (rule-based pi_theta) ---

@dataclass
class OpAmpSpec:
    '''Structured performance specification (pedagogical schema).'''
    name: str = "OTA1"
    vdd: float = 1.8
    vicm: float = 0.9
    target_gain_db: float = 60.0
    target_ugf_hz: float = 10e6
    load_pf: float = 1.0


def synthesize_pyspice_ota(spec: OpAmpSpec) -> str:
    gmid = 18.0 + np.clip(spec.target_gain_db / 20.0, 0, 6)
    ibias = max(5e-6, spec.target_ugf_hz / 5e11 * (1 + spec.load_pf))
    code = f'''
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import u_V, u_Ohm, u_pF, u_Hz

def build_{spec.name.lower()}():
    # Auto-generated OTA testbench skeleton (replace macros with MOS subckts)
    cir = Circuit('{spec.name}_AC')
    cir.V('dd', 'vdd', cir.gnd, {spec.vdd} @ u_V)
    cir.V('cm', 'vicm', cir.gnd, {spec.vicm} @ u_V)
    cir.SinusoidalVoltageSource('in_p', 'vip', 'vicm', dc_offset={spec.vicm} @ u_V, ac_magnitude=1 @ u_V)
    cir.SinusoidalVoltageSource('in_n', 'vin', 'vicm', dc_offset={spec.vicm} @ u_V, ac_magnitude=-1 @ u_V)
    cir.I('tail', 'vss', 'net_tail', {ibias:.3e})
    cir.C('load', 'vout', cir.gnd, {spec.load_pf} @ u_pF)
    cir.VCVS('gm_core', 'vout', cir.gnd, 'vip', 'vin', {10 ** (spec.target_gain_db / 20.0) * 1e-3:.6e})
    return cir

if __name__ == '__main__':
    ckt = build_{spec.name.lower()}()
    print(ckt)
'''
    return textwrap.dedent(code).strip()


spec = OpAmpSpec(name="TeachOTA", target_gain_db=62, target_ugf_hz=12e6, load_pf=2.0)
generated = synthesize_pyspice_ota(spec)
print(generated[:1400], "\n... [truncated]")


### 6.1.3 Surrogate metrics: THD, overshoot, settling

For periodic steady state, let discrete samples $y[n]$ have fundamental $f_0$ and sampling $f_s$. **Total harmonic distortion** is

$$
\mathrm{THD} = \frac{\sqrt{\sum_{h\ge 2} A_h^2}}{A_1},
$$

with $A_h$ the magnitude of the $h$-th harmonic bin.

For a step response, overshoot (when $y_\infty > y_0$) is

$$
O = \frac{y_{\max} - y_\infty}{y_\infty - y_0}.
$$

**Settling time** $T_s(\varepsilon)$ is the earliest time such that $|y(t) - y_\infty| \le \varepsilon |y_\infty - y_0|$ for all $t \ge T_s$.

These scalars can feed $e_\psi$ or a critic head that conditions the next code revision—mirroring **AnalogCoder-Pro** multimodal feedback.

---


In [ ]:
# --- Synthetic waveforms + AnalogCoder-Pro-style scalar diagnostics ---

def thd_from_signal(y: np.ndarray, f0_hz: float, fs_hz: float, max_h: int = 8) -> float:
    n = len(y)
    y = y - np.mean(y)
    spec = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs_hz)
    idx1 = int(np.argmin(np.abs(freqs - f0_hz)))
    a1 = np.abs(spec[idx1]) / (n / 2)
    num = 0.0
    for h in range(2, max_h + 1):
        fh = h * f0_hz
        ih = int(np.argmin(np.abs(freqs - fh)))
        ah = np.abs(spec[ih]) / (n / 2)
        num += ah**2
    return float(np.sqrt(num) / (a1 + 1e-12))


def step_response_metrics(t: np.ndarray, y: np.ndarray, eps: float = 0.02) -> Dict[str, float]:
    y0, yinf = float(y[0]), float(y[-1])
    ymax = float(np.max(y))
    overshoot = (ymax - yinf) / (abs(yinf - y0) + 1e-12)
    band = eps * abs(yinf - y0)
    settled = np.where(np.abs(y - yinf) <= band)[0]
    ts = float(t[settled[0]]) if len(settled) else float(t[-1])
    return {"overshoot": overshoot, "settling_s": ts, "y_inf": yinf}


fs = 2e7
T = 0.001
t = np.arange(0, T, 1 / fs)
f0 = 1e4
y_clean = np.sin(2 * np.pi * f0 * t)
y_dist = y_clean + 0.07 * np.sin(2 * np.pi * 2 * f0 * t) + 0.02 * RNG.normal(size=t.shape)

wn, zeta = 2 * np.pi * 5e4, 0.25
wd = wn * np.sqrt(max(1e-9, 1 - zeta**2))
y_step = 1.0 - np.exp(-zeta * wn * t) * (np.cos(wd * t) + (zeta * wn / (wd + 1e-12)) * np.sin(wd * t))

thd_clean = thd_from_signal(y_clean, f0, fs)
thd_dist = thd_from_signal(y_dist, f0, fs)
met = step_response_metrics(t, y_step)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=False)
ax[0].plot(t * 1e3, y_clean, color=GREEN, lw=1.0, alpha=0.9, label="clean")
ax[0].plot(t * 1e3, y_dist, color=AMBER, lw=1.0, alpha=0.85, label="distorted")
ax[0].set_xlim(0, 0.5)
ax[0].set_title("AnalogCoder-Pro view: time-domain waveforms")
ax[0].set_ylabel("y(t)")
ax[0].legend(loc="upper right")
ax[0].grid(True)

ax[1].plot(t * 1e6, y_step, color=ACCENT, lw=1.2)
ax[1].axhline(1.0, color="#8b949e", ls="--", lw=0.8)
ax[1].set_xlabel("time (µs)")
ax[1].set_title(
    f"Stability surrogate: step (overshoot={met['overshoot']:.2%}, Ts≈{met['settling_s']*1e6:.2f} µs)"
)
ax[1].grid(True)
plt.tight_layout()
plt.show()

print(f"THD (clean) ≈ {thd_clean*100:.3f}%  |  THD (distorted) ≈ {thd_dist*100:.3f}%")

win = int(fs * 0.0002)
seg = y_dist[:win] - np.mean(y_dist[:win])
spec = np.fft.rfft(seg * np.hanning(len(seg)))
freqs = np.fft.rfftfreq(len(seg), d=1 / fs)
figp = go.Figure()
figp.add_trace(
    go.Scatter(x=freqs / 1e3, y=20 * np.log10(np.abs(spec) + 1e-12), mode="lines", name="|Y(f)| dB")
)
figp.update_layout(
    title="Frequency-domain view of distortion (pedagogical FFT)",
    xaxis_title="f (kHz)",
    yaxis_title="dB",
    paper_bgcolor=DARK_BG,
    plot_bgcolor="#161b22",
    font=dict(color="#c9d1d9"),
)
figp.show()


## 6.2 AnalogSAGE — Stratified Self-Evolving Memory

**AnalogSAGE** organizes learning from closed-loop design into **three banks** with different *scope* and *volatility*:

1. **Evolution memory** $\mathcal{M}_E$: cross-task priors (“in this PDK, prefer folded-cascode when PSRR $> 80$ dB”).
2. **Introspective memory** $\mathcal{M}_I$: *within-task* post-mortems after failed simulations (local gradient-free “what went wrong”).
3. **Fusion memory** $\mathcal{M}_F$: **compressed** chain-of-thought traces—abstractions of long reasoning paths.

### 6.2.1 Write dynamics

Let $\mathcal{H}_t$ be the full interaction transcript up to $t$. Stratified extraction:

$$
\mathcal{M}_E \leftarrow \mathcal{U}_E(\mathcal{M}_E, \mathrm{Ext}_E(\mathcal{H}_t)), \quad
\mathcal{M}_I \leftarrow \mathcal{U}_I(\mathcal{M}_I, \mathrm{Ext}_I(\mathcal{H}_t)).
$$

Fusion is a **compression map** (summarization / clustering) with bottleneck:

$$
\mathcal{M}_F \leftarrow \mathcal{C}_\psi(\mathcal{M}_F, \mathrm{CoT}_t), \qquad
I(\mathrm{CoT}_t; \mathcal{M}_F^{\mathrm{new}}) \le I(\mathrm{CoT}_t; \mathrm{CoT}_t),
$$

i.e. stored fusion entries carry **fewer bits** than raw traces but preserve task-relevant sufficiency under retrieval.

### 6.2.2 Retrieval as gated attention

Query $q_t$ (current subgoal embedding). A simple fusion:

$$
m_t = \mathrm{Agg}\Big(
\mathrm{TopK}(q_t, \mathcal{M}_E),
\mathrm{TopK}(q_t, \mathcal{M}_I),
\mathrm{TopK}(q_t, \mathcal{M}_F)
\Big).
$$

We implement **hash-seeded pseudo-embeddings** (deterministic, offline-friendly) and visualize **bank growth** + **information flow**.

---


In [ ]:
# --- Stratified memory implementation (numpy + deterministic embeddings) ---

def _embed(text: str, dim: int = 32) -> np.ndarray:
    h = hashlib.sha256(text.encode()).digest()
    rng = np.random.default_rng(int.from_bytes(h[:8], "little"))
    v = rng.normal(size=dim)
    return v / (np.linalg.norm(v) + 1e-9)


@dataclass
class MemoryRecord:
    text: str
    vec: np.ndarray
    meta: Dict[str, Any] = field(default_factory=dict)


class StratifiedMemory:
    def __init__(self, dim: int = 32):
        self.dim = dim
        self.evolution: List[MemoryRecord] = []
        self.introspective: List[MemoryRecord] = []
        self.fusion: List[MemoryRecord] = []

    def add_evolution(self, text: str, **meta):
        self.evolution.append(MemoryRecord(text, _embed(text, self.dim), meta))

    def add_introspective(self, text: str, **meta):
        self.introspective.append(MemoryRecord(text, _embed(text, self.dim), meta))

    def add_fusion(self, cot_steps: Sequence[str], max_chars: int = 220):
        joined = " | ".join(cot_steps)
        summary = joined if len(joined) <= max_chars else joined[: max_chars - 3] + "..."
        self.fusion.append(MemoryRecord(summary, _embed(summary, self.dim), {"n_steps": len(cot_steps)}))

    def _topk(self, query: str, bank: List[MemoryRecord], k: int) -> List[Tuple[float, MemoryRecord]]:
        q = _embed(query, self.dim)
        scored = [(float(np.dot(q, r.vec)), r) for r in bank]
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:k]

    def retrieve(self, query: str, k: int = 3) -> Dict[str, List[MemoryRecord]]:
        return {
            "evolution": [r for _, r in self._topk(query, self.evolution, k)],
            "introspective": [r for _, r in self._topk(query, self.introspective, k)],
            "fusion": [r for _, r in self._topk(query, self.fusion, k)],
        }

    def sizes(self) -> Dict[str, int]:
        return {
            "evolution": len(self.evolution),
            "introspective": len(self.introspective),
            "fusion": len(self.fusion),
        }


mem = StratifiedMemory(dim=48)
mem.add_evolution("180nm thin-oxide PMOS: keep Vsg < 1.5 V for HCI budget", pdk="generic180")
mem.add_evolution("Current mirrors: match L, use integer W ratios for reproducibility", block="bias")
mem.add_introspective("Iter 1: phase margin 38° — increase Miller cap or raise tail Ibias", task="OTA_PM60")
mem.add_introspective("Ringing in transient — check zero in RHS, trial 150 fF load cap", task="OTA_PM60")
mem.add_fusion(
    [
        "Parse spec → folded-cascode template",
        "AC sim → PM 38° (fail spec 60°)",
        "Add Cm=120 fF → PM 62° (pass)",
        "Corner slow/nom/fast sweep",
    ]
)

q = "How to fix low phase margin on folded-cascode OTA?"
ret = mem.retrieve(q, k=2)
print("Query:", q)
for layer, recs in ret.items():
    print(f"\n[{layer}]")
    for r in recs:
        print(" -", r.text[:120])
print("\nBank sizes:", mem.sizes())


In [ ]:
# --- Memory evolution + Sankey view of stratified interactions ---

def design_iteration_loop(n: int = 8) -> Tuple[StratifiedMemory, List[Dict[str, int]]]:
    M = StratifiedMemory(dim=40)
    hist = []
    for it in range(1, n + 1):
        pm = 30 + it * 4 + int(RNG.integers(-2, 3))
        if pm < 45:
            M.add_introspective(f"iter {it}: PM={pm}° — bump Cm or Ibias", iter=it, pm=pm)
            M.add_fusion([f"iter{it}: ac_sim PM={pm}", "proposal: Cm += 40 fF", "retry AC"])
        else:
            M.add_evolution(f"iter {it}: converged PM={pm}° — archive sizing recipe", pm=pm)
            M.add_fusion([f"iter{it}: success PM={pm}", "freeze sizes", "run corners"])
        hist.append(M.sizes())
    return M, hist


M2, series = design_iteration_loop(10)
sizes = np.array([[d["evolution"], d["introspective"], d["fusion"]] for d in series])

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(1, len(series) + 1)
ax.plot(x, sizes[:, 0], "o-", color=GREEN, label=r"$\mathcal{M}_E$ evolution")
ax.plot(x, sizes[:, 1], "s-", color=AMBER, label=r"$\mathcal{M}_I$ introspective")
ax.plot(x, sizes[:, 2], "^-", color=PURPLE, label=r"$\mathcal{M}_F$ fusion")
ax.set_xlabel("Design iteration")
ax.set_ylabel("Record count")
ax.set_title("Stratified memory growth (toy self-evolving loop)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

labels = ["Task", "Simulator", r"$\mathcal{M}_E$", r"$\mathcal{M}_I$", r"$\mathcal{M}_F$", "Next action"]
source = [0, 1, 1, 1, 2, 3, 4]
target = [1, 2, 3, 4, 5, 5, 5]
value = [10, 3, 4, 3, 3, 4, 3]
figs = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                label=labels,
                pad=18,
                thickness=18,
                color="#21262d",
                line=dict(color="#30363d", width=1),
            ),
            link=dict(
                source=source,
                target=target,
                value=value,
                color="rgba(88,166,255,0.35)",
            ),
        )
    ]
)
figs.update_layout(
    title="Oracle feedback → stratified writes → next policy step",
    font=dict(size=12),
    paper_bgcolor=DARK_BG,
)
figs.show()


## 6.3 GENIE-ASI (Sony AI) — Training-Free Structure + Few-Shot Code

GENIE-ASI combines **program synthesis** (Python that manipulates netlist objects) with **inductive biases** from analog topology. A useful abstraction is the **device graph** $\mathcal{G}=(\mathcal{V},\mathcal{E})$: nodes are MOSFETs, resistors, capacitors; edges are **shared nets**.

**Subcircuit identification** (training-free) selects a pattern $P_b$ for block $b$ from a library $\mathcal{B}$:

$$
\hat{b} = \arg\max_{b \in \mathcal{B}} \; \mathrm{score}(P_b, \mathcal{G}).
$$

We implement **deterministic** detectors for:

- **Differential pair**: two MOS devices sharing **source** net, distinct **gates** (small-signal inputs).
- **NMOS current mirror**: devices sharing **gate** net, reference device in **diode** configuration ($d \equiv g$).

Few-shot LLMs (in deployment) emit code that calls such detectors recursively—here we show the **typed primitives**.

---


In [ ]:
# --- SPICE MOS lines -> graph primitives + block detection ---

@dataclass
class MosDevice:
    name: str
    model: str
    d: str
    g: str
    s: str
    b: str


def parse_mos_lines(netlist: str) -> List[MosDevice]:
    devs: List[MosDevice] = []
    for line in netlist.splitlines():
        line = line.strip()
        if not line or line.startswith("*") or line.startswith("."):
            continue
        parts = line.split()
        if len(parts) >= 6 and parts[0].lower().startswith("m"):
            name, d, g, s, b, model = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
            devs.append(MosDevice(name, model, d, g, s, b))
    return devs


def detect_differential_pair(devs: List[MosDevice]) -> List[Tuple[str, str]]:
    pairs = []
    by_source: Dict[str, List[MosDevice]] = defaultdict(list)
    for m in devs:
        by_source[m.s].append(m)
    for snet, group in by_source.items():
        if len(group) < 2:
            continue
        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                a, b = group[i], group[j]
                if a.g != b.g:
                    pairs.append((a.name, b.name))
    return pairs


def detect_nmos_current_mirror(devs: List[MosDevice]) -> List[Tuple[str, str]]:
    mirrors = []
    gates: Dict[str, List[MosDevice]] = defaultdict(list)
    for m in devs:
        gates[m.g].append(m)
    for gnet, group in gates.items():
        if len(group) < 2:
            continue
        diode = [m for m in group if m.d == m.g]
        if not diode:
            continue
        ref = diode[0]
        for m in group:
            if m.name != ref.name:
                mirrors.append((ref.name, m.name))
    return mirrors


example_nl = "\n".join(
    [
        "* Input pair + simple mirror",
        "M1 outn inp net_tail vss nmos w=2u l=180n",
        "M2 outp inn net_tail vss nmos w=2u l=180n",
        "Mref n_gate n_gate vss vss nmos w=2u l=180n",
        "Mcopy net_ib n_gate vss vss nmos w=6u l=180n",
    ]
)

devices = parse_mos_lines(example_nl)
print("Devices:", [(d.name, d.d, d.g, d.s) for d in devices])
print("Differential pairs (names):", detect_differential_pair(devices))
print("NMOS mirrors (ref, copy):", detect_nmos_current_mirror(devices))


## 6.4 Schemato — Fine-Tuned Llama 3.1-8B for Netlist $\rightarrow$ `.asc`

**Schemato** addresses the *visualization gap*: SPICE is machine-friendly but **schematics** are human-friendly. A fine-tuned **Llama 3.1-8B** maps netlist tokens to **LTspice schematic files** (components + `WIRE` + `FLAG`).

Conceptually:

$$
\text{SPICE} \;\xrightarrow{\;\pi_\theta^{\text{Schemato}}\;}\; \text{ASC} = (\text{symbols}, \text{wires}, \text{annotations}).
$$

**Stages** in a production pipeline:

1. **Parse / normalize** — device records, model cards, subckt hierarchy.
2. **Graph layout** — planarity heuristics, symmetric placement for matched devices.
3. **ASC emission** — map to LTspice symbol orientation and grid snapping.
4. **Human-in-the-loop** — designer tweaks, then `.asc` round-trips with LTspice GUI.

Below: a **deterministic** micro-emitter (grid placement) illustrating IO shape—not the learned mapper.

---


In [ ]:
# --- Conceptual netlist -> .asc (heuristic grid; not the Schemato weights) ---

def netlist_to_asc_demo(netlist: str, title: str = "demo") -> str:
    devs = []
    for line in netlist.splitlines():
        line = line.strip()
        if not line or line.startswith("*"):
            continue
        if line.lower().startswith("m"):
            tok = line.split()
            name = tok[0]
            devs.append((name, tok[1:5]))

    lines_out = ["Version 4", "SymbolType BLOCK", f"TEXT 0 -40 Left 0 ! {title}"]
    x0, y0 = 80, 80
    dy = 80
    nodes_xy: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
    for i, (name, pins) in enumerate(devs):
        xi, yi = x0, y0 + i * dy
        lines_out.append(f"SYMBOL nmos  {xi} {yi} R0")
        lines_out.append("WINDOW 0 32 32 Left 0")
        lines_out.append(f"SYMATTR InstName {name}")
        for pi, pin in enumerate(["drain", "gate", "source", "bulk"]):
            nodes_xy[pins[pi]].append((xi + 40, yi + 10 + pi * 8))

    for net, pts in nodes_xy.items():
        if len(pts) < 2:
            continue
        x1, y1 = pts[0]
        for (x2, y2) in pts[1:]:
            lines_out.append(f"WIRE {x1} {y1} {x2} {y2}")

    lines_out.append("FLAG 0 0 0")
    return "\n".join(lines_out)


asc_text = netlist_to_asc_demo(example_nl, title="TeachSchemato")
print(asc_text[:900], "\n...")

stages = ["Parse", "Build $\\mathcal{G}$", "Layout", "Emit ASC", "Designer edit"]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=list(range(len(stages))),
        y=[0] * len(stages),
        mode="markers+text",
        text=stages,
        textposition="top center",
        marker=dict(size=18, color=ACCENT),
    )
)
for i in range(len(stages) - 1):
    fig.add_shape(
        type="line",
        x0=i,
        y0=0,
        x1=i + 1,
        y1=0,
        line=dict(color="#8b949e", width=2),
    )
fig.update_layout(
    title="Schemato-style conversion pipeline (conceptual)",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 0.5]),
    height=260,
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor=DARK_BG,
)
fig.show()


## 6.5 LayoutCopilot — LLM Multi-Agent Collaborative Layout

**LayoutCopilot** frames interactive analog layout as **multi-agent collaboration** under **physical design oracles** (DRC, LVS, PEX). Typical roles:

| Agent | Action space | Objective |
|-------|--------------|-----------|
| **Placement** $A_P$ | device coordinates, symmetry groups | matching, gradient / STI proximity |
| **Routing** $A_R$ | guides, shielding, layer choices | min parasitics, EM margin |
| **Verification** $A_V$ | rule decks, annotations | reduce violation set $\mathcal{V}_t \rightarrow \varnothing$ |

A simplified message pass:

$$
m_{t+1}^{(P)} = f_P(x_t, \mathcal{V}_t, m_t^{(R)}), \qquad
m_{t+1}^{(R)} = f_R(\mathcal{N}_t, m_t^{(P)}),
$$

with shared **violation energy** $E(\mathcal{V}_t)$ driving consensus. Below, $E$ is a toy pairwise-distance penalty mimicking minimum-spacing stress.

---


In [ ]:
# --- Toy LayoutCopilot: placer + router agents minimize spacing violations ---

@dataclass
class Device:
    name: str
    pos: np.ndarray


def drc_violation(devs: List[Device], min_sep: float = 1.2) -> float:
    stress = 0.0
    for i in range(len(devs)):
        for j in range(i + 1, len(devs)):
            d = np.linalg.norm(devs[i].pos - devs[j].pos)
            stress += max(0.0, min_sep - d) ** 2
    return float(stress)


def agent_placer(devs: List[Device], grad_scale: float = 0.35) -> None:
    n = len(devs)
    forces = [np.zeros(2) for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            delta = devs[i].pos - devs[j].pos
            dist = np.linalg.norm(delta) + 1e-6
            if dist < 1.6:
                push = (delta / dist) * (1.6 - dist)
                forces[i] += push
                forces[j] -= push
    for i in range(n):
        devs[i].pos += grad_scale * forces[i]


def agent_router(devs: List[Device], anchor: np.ndarray) -> None:
    for d in devs:
        d.pos = 0.92 * d.pos + 0.08 * anchor


devs = [
    Device("M1", np.array([0.0, 0.0])),
    Device("M2", np.array([0.4, 0.1])),
    Device("M3", np.array([0.2, 0.35])),
]
traj = []
viol = []
for t in range(40):
    v = drc_violation(devs)
    viol.append(v)
    traj.append(np.stack([d.pos.copy() for d in devs]))
    if v < 1e-3:
        break
    agent_placer(devs)
    agent_router(devs, anchor=np.array([1.0, 0.2]))

arr = np.stack(traj)
fig = go.Figure()
colors = [GREEN, AMBER, PURPLE]
for i, d in enumerate(devs):
    fig.add_trace(
        go.Scatter(
            x=arr[:, i, 0],
            y=arr[:, i, 1],
            mode="lines+markers",
            name=d.name,
            line=dict(color=colors[i % len(colors)]),
            marker=dict(size=6),
        )
    )
fig.update_layout(
    title="LayoutCopilot toy: collaborative relaxation of DRC stress",
    xaxis_title="x (arb.)",
    yaxis_title="y (arb.)",
    height=480,
    paper_bgcolor=DARK_BG,
)
fig.show()

fig2 = go.Figure(go.Scatter(y=viol, mode="lines", line=dict(color=ACCENT)))
fig2.update_layout(
    title="Violation energy vs iteration",
    xaxis_title="t",
    yaxis_title="stress",
    paper_bgcolor=DARK_BG,
)
fig2.show()


## 6.6 Comparative Synthesis

| Framework | Primary artifact | Learning signal | Key oracle | Multi-agent? |
|-----------|------------------|-----------------|------------|--------------|
| **AnalogCoder / Pro** | Simulator code $c$ | SFT / RL + multimodal critic | $\mathcal{S}(c)$, waveform features | Optional |
| **AnalogSAGE** | Evolving $\mathcal{M}_E,\mathcal{M}_I,\mathcal{M}_F$ | Self-evolution + retrieval | Simulation + reflection | Controller + memory |
| **GENIE-ASI** | Python over netlists | Few-shot program priors | Pattern checks / tests | Tool-using LLM |
| **Schemato** | `.asc` schematic | SFT Llama 3.1-8B | Visual + LTspice | Usually single model |
| **LayoutCopilot** | Layout edits | LLM policies + UI loop | DRC/LVS/PEX | **Yes** (placement/routing/verify) |

**Closing thought:** specialized analog agents succeed when **actions are typed**, **oracles are real**, and **memory is stratified** so cross-task priors do not drown task-specific failures—precisely the decomposition this chapter exercised in miniature.

---
